In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform

In [2]:
import json

In [3]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
data = pd.read_csv('../data/loan_data.csv')

In [5]:
data.head(15)

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
5,21.0,female,High School,12951.0,0,OWN,2500.0,VENTURE,7.14,0.19,2.0,532,No,1
6,26.0,female,Bachelor,93471.0,1,RENT,35000.0,EDUCATION,12.42,0.37,3.0,701,No,1
7,24.0,female,High School,95550.0,5,RENT,35000.0,MEDICAL,11.11,0.37,4.0,585,No,1
8,24.0,female,Associate,100684.0,3,RENT,35000.0,PERSONAL,8.90,0.35,2.0,544,No,1
9,21.0,female,High School,12739.0,0,OWN,1600.0,VENTURE,14.74,0.13,3.0,640,No,1


In [6]:
data['loan_status'].value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [7]:
def categorize_and_split(df : pd.DataFrame):

    X = df.drop('loan_status', axis = 1)
    y = df['loan_status']

    categorical_features = X.select_dtypes(
    include = ["object", "category", "bool"]).columns

    X[categorical_features] = X[categorical_features].astype('category')

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify=y, random_state=42)  # changed: fixed seed so the test split is the same every run

    return X_train, X_test, y_train, y_test
    

In [8]:
def scale_weights(y):
    neg = (y==0).sum()
    pos = (y==1).sum()
    weight = neg/pos

    return weight

In [9]:
#def undersample_imbalanced(X_train, y_train):

   # rus = RandomUnderSampler(sampling_strategy =2/3)
   # X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

   # print("Before:", y_train.value_counts().to_dict())
   # print("After:", y_train_res.value_counts().to_dict())

   # return X_train_res, y_train_res
    

In [10]:
def search_and_train(X, y, scale_weights):

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, stratify = y, random_state=42)

    weight = scale_weights(y_train)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    param_dist = {
        'n_estimators' : randint(100, 500),
        'max_depth': randint(3, 10),
        'min_child_weight' : randint(1, 20),
        'learning_rate': uniform(0.01, 0.3),
        'subsample': uniform(0.6 , 0.4) }
                                            

    search = RandomizedSearchCV(
        xgb.XGBClassifier(
        objective = 'binary:logistic',
        learning_rate = 0.05,
        scale_pos_weight = weight,
        tree_method = 'hist',
        enable_categorical = True
        ),

        param_distributions = param_dist,
        n_iter = 15,
        scoring = 'average_precision',
        cv = cv,
        return_train_score = True,
        random_state = 42,
        n_jobs = -1)  # added: run CV fits in parallel on all cores for speed

    results = search.fit(X_train, y_train)

    cols = ["params", 'mean_test_score', 'std_test_score', 'mean_train_score']

    results_df = pd.DataFrame(search.cv_results_)[cols].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

    best_model = search.best_estimator_

    return results_df, best_model , X_val, y_val
    

In [11]:
splitX_train, splitX_test, splity_train, splity_test = categorize_and_split(data)

In [12]:
summary_df, fit_model, X_val, y_val = search_and_train(splitX_train, splity_train, scale_weights)

In [13]:
summary_df.shape

(15, 4)

In [14]:
dtype_dict = {col :str(dtype) for col, dtype in splitX_train.dtypes.items()}

In [15]:
with open("feature_dtypes.json", "w") as f:
    json.dump(dtype_dict, f)

In [16]:
fit_model.save_model('xgboost_trained.json')  # changed: XGBClassifier has save_model, not save

In [17]:
from sklearn.metrics import classification_report  # added: import report function
X_tr, y_tr = splitX_train.drop(X_val.index), splity_train.drop(y_val.index)  # added: actual train rows (excludes the val split made inside search_and_train)
for name, X_, y_ in [('TRAIN', X_tr, y_tr), ('VALIDATION', X_val, y_val), ('TEST', splitX_test, splity_test)]:  # added: loop over the three splits
    print(f'===== {name} =====')  # added: header for each split
    print(classification_report(y_, fit_model.predict(X_), digits=4))  # added: precision/recall/f1 per class


===== TRAIN =====


              precision    recall  f1-score   support

           0     0.9949    0.9497    0.9718     23800
           1     0.8480    0.9829    0.9105      6800

    accuracy                         0.9571     30600
   macro avg     0.9215    0.9663    0.9411     30600
weighted avg     0.9623    0.9571    0.9581     30600

===== VALIDATION =====
              precision    recall  f1-score   support

           0     0.9656    0.9229    0.9438      4200
           1     0.7662    0.8850    0.8213      1200

    accuracy                         0.9144      5400
   macro avg     0.8659    0.9039    0.8826      5400
weighted avg     0.9213    0.9144    0.9166      5400

===== TEST =====
              precision    recall  f1-score   support

           0     0.9706    0.9260    0.9478      7000
           1     0.7769    0.9020    0.8348      2000

    accuracy                         0.9207      9000
   macro avg     0.8738    0.9140    0.8913      9000
weighted avg     0.9276    0.9207 

In [18]:
import os  # added: for creating the results folder
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score  # added: extra summary metrics
os.makedirs('results', exist_ok=True)  # added: folder to store outputs
summary_df.to_csv('results/cv_search_results.csv', index=False)  # added: all 15 hyperparameter settings with mean/std CV scores
rows = []  # added: collects per-split metrics
with open('results/classification_reports.txt', 'w') as f:  # added: text file with all reports
    for name, X_, y_ in [('TRAIN', X_tr, y_tr), ('VALIDATION', X_val, y_val), ('TEST', splitX_test, splity_test)]:  # added: loop over splits
        pred, prob = fit_model.predict(X_), fit_model.predict_proba(X_)[:, 1]  # added: class predictions and class-1 probabilities
        f.write(f'===== {name} =====' + chr(10) + classification_report(y_, pred, digits=4) + chr(10))  # added: write report to file
        rep = classification_report(y_, pred, output_dict=True)  # added: report as dict for the table
        rows.append({'split': name, 'accuracy': accuracy_score(y_, pred), 'precision_1': rep['1']['precision'], 'recall_1': rep['1']['recall'], 'f1_1': rep['1']['f1-score'], 'roc_auc': roc_auc_score(y_, prob), 'pr_auc': average_precision_score(y_, prob)})  # added: one summary row per split
metrics_df = pd.DataFrame(rows)  # added: summary table
metrics_df.to_csv('results/split_metrics.csv', index=False)  # added: save summary table
json.dump(fit_model.get_params(), open('results/best_params.json', 'w'), indent=2, default=str)  # added: best model hyperparameters
print(summary_df.head(15))  # added: show all search candidates
metrics_df  # added: display summary table


                                               params  mean_test_score  \
0   {'learning_rate': 0.12236203565420874, 'max_de...         0.935067   
1   {'learning_rate': 0.24555278841790407, 'max_de...         0.934989   
2   {'learning_rate': 0.06502135295603015, 'max_de...         0.934834   
3   {'learning_rate': 0.02692347370813008, 'max_de...         0.934249   
4   {'learning_rate': 0.19355586841671385, 'max_de...         0.933362   
5   {'learning_rate': 0.029515477895583855, 'max_d...         0.932331   
6   {'learning_rate': 0.30087538832936755, 'max_de...         0.931496   
7   {'learning_rate': 0.26985284373248053, 'max_de...         0.930973   
8   {'learning_rate': 0.150028867974394, 'max_dept...         0.930829   
9   {'learning_rate': 0.1890550473839461, 'max_dep...         0.930814   
10  {'learning_rate': 0.13754676234737342, 'max_de...         0.930773   
11  {'learning_rate': 0.06879485872574355, 'max_de...         0.929685   
12  {'learning_rate': 0.28279612062363

,split,accuracy,precision_1,recall_1,f1_1,roc_auc,pr_auc
0,TRAIN,0.957059,0.848008,0.982941,0.910503,0.994985,0.982652
1,VALIDATION,0.914444,0.766234,0.885000,0.821346,0.975680,0.932961
2,TEST,0.920667,0.776916,0.902000,0.834799,0.978398,0.939600


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score  # added: per-class metrics for the run table
best_params = {k: v for k, v in fit_model.get_params().items() if k != 'scale_pos_weight'}  # added: best hyperparameters from the search (weight recomputed per run)
X_all = data.drop('loan_status', axis=1)  # added: features
y_all = data['loan_status']  # added: target
cat_cols = X_all.select_dtypes(include=['object', 'category', 'bool']).columns  # added: categorical columns
X_all[cat_cols] = X_all[cat_cols].astype('category')  # added: same categorical handling as categorize_and_split
run_rows, report_rows = [], []  # added: per-run summary rows and full per-class report rows
for run in range(5):  # added: 5 runs, each with a different random split
    X_trv, X_te, y_trv, y_te = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=run)  # added: different test split per run
    X_t, X_v, y_t, y_v = train_test_split(X_trv, y_trv, test_size=0.15, stratify=y_trv, random_state=run)  # added: different train/val split per run
    model = xgb.XGBClassifier(**best_params, scale_pos_weight=scale_weights(y_t))  # added: same best settings, weight from this run's train set
    model.fit(X_t, y_t)  # added: train on this run's train set
    for split_name, X_, y_ in [('train', X_t, y_t), ('val', X_v, y_v), ('test', X_te, y_te)]:  # added: score each split
        pred, prob = model.predict(X_), model.predict_proba(X_)[:, 1]  # added: predictions and class-1 probabilities
        run_rows.append({'run': run + 1, 'split': split_name, 'accuracy': accuracy_score(y_, pred), 'precision_1': precision_score(y_, pred), 'recall_1': recall_score(y_, pred), 'f1_1': f1_score(y_, pred), 'roc_auc': roc_auc_score(y_, prob), 'pr_auc': average_precision_score(y_, prob)})  # added: one row per run per split
        rep = classification_report(y_, pred, output_dict=True)  # added: full report as dict
        for label in ['0', '1', 'macro avg', 'weighted avg']:  # added: flatten the report rows
            report_rows.append({'run': run + 1, 'split': split_name, 'class': label, 'precision': rep[label]['precision'], 'recall': rep[label]['recall'], 'f1': rep[label]['f1-score'], 'support': rep[label]['support']})  # added: one row per class per split per run
runs_df = pd.DataFrame(run_rows).round(4)  # added: table of all runs
reports_df = pd.DataFrame(report_rows).round(4)  # added: classification reports of all runs as a table
metric_cols = ['accuracy', 'precision_1', 'recall_1', 'f1_1', 'roc_auc', 'pr_auc']  # added: metric columns
runs_summary = runs_df.groupby('split')[metric_cols].agg(['mean', 'std']).round(4).loc[['train', 'val', 'test']]  # added: mean and std across the 5 runs per split
runs_df.to_csv('results/runs_metrics.csv', index=False)  # added: 15 rows = 5 runs x 3 splits
reports_df.to_csv('results/runs_classification_reports.csv', index=False)  # added: full per-class reports for every run
runs_summary.to_csv('results/runs_summary_mean_std.csv')  # added: mean/std table
for split_name in ['train', 'val', 'test']:  # added: print one table per split
    print(f'===== {split_name.upper()} (5 runs) =====')  # added: header
    print(runs_df[runs_df['split'] == split_name].drop(columns='split').to_string(index=False))  # added: 5 rows, one per run
    print()  # added: blank line
runs_summary  # added: display mean and std per split


===== TRAIN (5 runs) =====
 run  accuracy  precision_1  recall_1   f1_1  roc_auc  pr_auc
   1    0.9576       0.8480    0.9860 0.9118   0.9951  0.9830
   2    0.9558       0.8454    0.9807 0.9080   0.9950  0.9827
   3    0.9562       0.8454    0.9826 0.9089   0.9950  0.9828
   4    0.9579       0.8502    0.9841 0.9123   0.9950  0.9826
   5    0.9580       0.8511    0.9832 0.9124   0.9952  0.9834

===== VAL (5 runs) =====
 run  accuracy  precision_1  recall_1   f1_1  roc_auc  pr_auc
   1    0.9189       0.7745    0.8958 0.8308   0.9783  0.9398
   2    0.9128       0.7587    0.8908 0.8195   0.9756  0.9318
   3    0.9163       0.7679    0.8933 0.8259   0.9782  0.9388
   4    0.9211       0.7833    0.8917 0.8340   0.9782  0.9385
   5    0.9113       0.7551    0.8892 0.8167   0.9760  0.9336

===== TEST (5 runs) =====
 run  accuracy  precision_1  recall_1   f1_1  roc_auc  pr_auc
   1    0.9197       0.7756    0.8985 0.8325   0.9779  0.9390
   2    0.9188       0.7729    0.8985 0.8310   0.978

accuracy         precision_1         recall_1            f1_1          \
          mean     std        mean     std     mean     std    mean     std   
split                                                                         
train   0.9571  0.0010      0.8480  0.0026   0.9833  0.0019  0.9107  0.0021   
val     0.9161  0.0041      0.7679  0.0115   0.8922  0.0025  0.8254  0.0073   
test    0.9182  0.0016      0.7713  0.0036   0.8981  0.0036  0.8298  0.0032   

      roc_auc          pr_auc          
         mean     std    mean     std  
split                                  
train  0.9951  0.0001  0.9829  0.0003  
val    0.9773  0.0013  0.9365  0.0036  
test   0.9781  0.0005  0.9381  0.0017